# Desafio de IA: Engenharia de Prompt e Privacidade de Dados

## Objetivos

Ao final da atividade, você deverá ser capaz de:

- identificar dados pessoais e campos potencialmente sensíveis;
- aplicar tokenização, redação, mascaramento e generalização;
- validar se a anonimização removeu os identificadores;
- analisar avaliações e comentários com Python;
- criar prompts claros usando apenas dados anonimizados ou agregados;
- revisar criticamente respostas produzidas por uma IA.

> **Regra de segurança:** não envie o arquivo original para ferramentas públicas de IA. Nesta atividade, o tratamento e a análise serão feitos no Colab. Somente dados anonimizados ou resultados agregados poderão ser utilizados na etapa de prompts.

## 1. Entendendo o cenário

O arquivo contém:

- `CustomerName`: nome do cliente;
- `Email`: endereço de e-mail;
- `IPAddress`: endereço IP;
- `Feedback`: comentário;
- `Rating`: avaliação numérica.

Antes da análise, precisamos perguntar:

1. Quais colunas identificam diretamente uma pessoa?
2. Quais campos podem conter identificadores escondidos?
3. Quais informações são necessárias para responder às perguntas do negócio?
4. O que pode ser removido ou transformado sem prejudicar a análise?

## 2. Preparar o ambiente

Execute a célula abaixo para importar as bibliotecas utilizadas.

In [ ]:
import pandas as pd
import re
from collections import Counter
from pathlib import Path

pd.set_option("display.max_colwidth", 120)
print("Ambiente preparado.")

Ambiente preparado.


## 3. Carregar o arquivo CSV

Ao executar a próxima célula no Google Colab:

1. clique em **Escolher arquivos**;
2. selecione `Copy of customer_feedback.csv` ou `Copy of feedback.csv`;
3. aguarde a confirmação do nome do arquivo.

In [ ]:
try:
    from google.colab import files
    arquivos = files.upload()
    nome_arquivo = next(iter(arquivos))
except ImportError:
    # Permite testar o notebook fora do Colab.
    candidatos = [
        "Copy of customer_feedback.csv",
        "Copy of feedback.csv",
        "customer_feedback.csv",
        "feedback.csv",
    ]
    nome_arquivo = next((x for x in candidatos if Path(x).exists()), None)
    if nome_arquivo is None:
        raise FileNotFoundError("Coloque um dos arquivos CSV na mesma pasta do notebook.")

df_original = pd.read_csv(nome_arquivo)
print(f"Arquivo carregado: {nome_arquivo}")
print(f"Linhas: {df_original.shape[0]} | Colunas: {df_original.shape[1]}")
display(df_original.head())

Saving Copy of customer_feedback.csv to Copy of customer_feedback.csv
Arquivo carregado: Copy of customer_feedback.csv
Linhas: 15 | Colunas: 5


,CustomerName,Email,IPAddress,Feedback,Rating
0,Alice Johnson,alice.johnson@example.com,192.168.1.10,Excellent support!,5
1,Bob Williams,bob.williams@example.com,192.168.1.11,Very helpful staff.,4
2,Charlie Brown,charlie.brown@example.com,192.168.1.12,Average experience.,3
3,Diana Prince,diana.prince@example.com,192.168.1.13,Amazing product quality.,5
4,Evan Smith,evan.smith@example.com,192.168.1.14,Quick resolution of issues.,4


## 4. Conhecer os dados antes de transformá-los

Não comece alterando o arquivo. Primeiro, observe:

- nomes das colunas;
- tipos de dados;
- valores ausentes;
- possíveis identificadores;
- presença de dados pessoais dentro dos comentários.

In [ ]:
print("Colunas:", list(df_original.columns))
print("\nTipos de dados:")
display(df_original.dtypes.to_frame("tipo"))

print("\nValores ausentes:")
display(df_original.isna().sum().to_frame("quantidade"))

print("\nResumo das avaliações:")
display(df_original["Rating"].describe().to_frame())

Colunas: ['CustomerName', 'Email', 'IPAddress', 'Feedback', 'Rating']

Tipos de dados:


,tipo
CustomerName,object
Email,object
IPAddress,object
Feedback,object
Rating,int64



Valores ausentes:


,quantidade
CustomerName,0
Email,0
IPAddress,0
Feedback,0
Rating,0



Resumo das avaliações:


,Rating
count,15.0
mean,4.0
std,1.0
min,2.0
25%,3.0
50%,4.0
75%,5.0
max,5.0


### Classifique as colunas

Preencha a justificativa antes de executar a próxima etapa.

| Coluna | Classificação sugerida | Justificativa |
|---|---|---|
| CustomerName | identificador direto | permite reconhecer a pessoa |
| Email | identificador direto | endereço associado à pessoa |
| IPAddress | identificador técnico | pode contribuir para identificação |
| Feedback | campo a revisar | texto livre pode conter dados pessoais |
| Rating | dado analítico | necessário para a análise, mas não identifica isoladamente |

## 5. Procurar identificadores dentro do texto livre

Mesmo que a coluna se chame `Feedback`, um cliente pode escrever seu e-mail, telefone ou IP dentro do comentário. A função abaixo procura alguns padrões comuns.

> Uma expressão regular ajuda a localizar padrões, mas não garante que todos os dados pessoais serão encontrados. A revisão humana continua necessária.

In [ ]:
padroes = {
    "email": r"[\w\.-]+@[\w\.-]+\.\w+",
    "ip": r"\b(?:\d{1,3}\.){3}\d{1,3}\b",
    "telefone": r"\b(?:\+?\d{1,3}[\s-]?)?(?:\(?\d{2}\)?[\s-]?)?\d{4,5}[\s-]?\d{4}\b",
}

def localizar_padroes(texto):
    texto = str(texto)
    encontrados = []
    for nome, padrao in padroes.items():
        if re.search(padrao, texto):
            encontrados.append(nome)
    return ", ".join(encontrados) if encontrados else "nenhum"

df_original["PadroesEncontrados"] = df_original["Feedback"].apply(localizar_padroes)
display(df_original[["Feedback", "PadroesEncontrados"]])

,Feedback,PadroesEncontrados
0,Excellent support!,nenhum
1,Very helpful staff.,nenhum
2,Average experience.,nenhum
3,Amazing product quality.,nenhum
4,Quick resolution of issues.,nenhum
5,Satisfied with the service.,nenhum
6,Will recommend to others.,nenhum
7,Could improve delivery time.,nenhum
8,Exceptional customer care.,nenhum
9,Not satisfied with the experience.,nenhum


## 6. Criar uma cópia para anonimização

Nunca trabalhe diretamente sobre o único arquivo original. A cópia permite comparar, corrigir e auditar as transformações.

In [ ]:
df_anonimo = df_original.drop(columns=["PadroesEncontrados"], errors="ignore").copy()
print("Cópia criada. O DataFrame original foi preservado.")

Cópia criada. O DataFrame original foi preservado.


## 7. Aplicar técnicas de anonimização

Nesta solução:

- **tokenização:** o nome é substituído por `Cliente001`, `Cliente002` etc.;
- **redação:** o e-mail é substituído por `[EMAIL_REMOVIDO]`;
- **generalização:** o IP mantém somente os dois primeiros blocos;
- **generalização analítica:** a avaliação recebe uma categoria;
- **higienização de texto:** e-mails, IPs e telefones encontrados no feedback são removidos.

In [ ]:
def tokenizar_clientes(serie):
    return [f"Cliente{i:03d}" for i in range(1, len(serie) + 1)]

def generalizar_ip(ip):
    partes = str(ip).split(".")
    if len(partes) == 4:
        return f"{partes[0]}.{partes[1]}.xxx.xxx"
    return "[IP_GENERALIZADO]"

def higienizar_texto(texto):
    texto = str(texto)
    texto = re.sub(padroes["email"], "[EMAIL_REMOVIDO]", texto)
    texto = re.sub(padroes["ip"], "[IP_REMOVIDO]", texto)
    texto = re.sub(padroes["telefone"], "[TELEFONE_REMOVIDO]", texto)
    return texto

def categoria_avaliacao(nota):
    if nota <= 2:
        return "Baixa"
    if nota == 3:
        return "Média"
    return "Alta"

df_anonimo["CustomerName"] = tokenizar_clientes(df_anonimo["CustomerName"])
df_anonimo["Email"] = "[EMAIL_REMOVIDO]"
df_anonimo["IPAddress"] = df_anonimo["IPAddress"].apply(generalizar_ip)
df_anonimo["Feedback"] = df_anonimo["Feedback"].apply(higienizar_texto)
df_anonimo["RatingCategory"] = df_anonimo["Rating"].apply(categoria_avaliacao)

display(df_anonimo.head())

,CustomerName,Email,IPAddress,Feedback,Rating,RatingCategory
0,Cliente001,[EMAIL_REMOVIDO],192.168.xxx.xxx,Excellent support!,5,Alta
1,Cliente002,[EMAIL_REMOVIDO],192.168.xxx.xxx,Very helpful staff.,4,Alta
2,Cliente003,[EMAIL_REMOVIDO],192.168.xxx.xxx,Average experience.,3,Média
3,Cliente004,[EMAIL_REMOVIDO],192.168.xxx.xxx,Amazing product quality.,5,Alta
4,Cliente005,[EMAIL_REMOVIDO],192.168.xxx.xxx,Quick resolution of issues.,4,Alta


## 8. Validar a anonimização

Anonimizar não é apenas executar uma função. Precisamos verificar se:

- nomes e e-mails originais não aparecem no resultado;
- IPs completos foram removidos;
- os comentários não contêm os padrões pesquisados;
- os dados necessários para análise continuam disponíveis.

In [ ]:
def contar_valores_originais_presentes(original, anonimo, coluna):
    valores_originais = set(original[coluna].dropna().astype(str))
    valores_anonimos = set(anonimo[coluna].dropna().astype(str))
    return len(valores_originais.intersection(valores_anonimos))

validacao = {
    "nomes_originais_restantes": contar_valores_originais_presentes(df_original, df_anonimo, "CustomerName"),
    "emails_originais_restantes": contar_valores_originais_presentes(df_original, df_anonimo, "Email"),
    "ips_completos_restantes": df_anonimo["IPAddress"].astype(str).str.match(r"^(?:\d{1,3}\.){3}\d{1,3}$").sum(),
    "emails_no_feedback": df_anonimo["Feedback"].str.contains(padroes["email"], regex=True, na=False).sum(),
    "ips_no_feedback": df_anonimo["Feedback"].str.contains(padroes["ip"], regex=True, na=False).sum(),
}

display(pd.Series(validacao, name="quantidade").to_frame())

if all(valor == 0 for valor in validacao.values()):
    print("Validação básica concluída: nenhum dos padrões verificados permaneceu.")
else:
    print("Atenção: revise os itens com quantidade maior que zero.")

,quantidade
nomes_originais_restantes,0
emails_originais_restantes,0
ips_completos_restantes,0
emails_no_feedback,0
ips_no_feedback,0


Validação básica concluída: nenhum dos padrões verificados permaneceu.


## 9. Salvar e baixar o arquivo anonimizado

O arquivo original permanece inalterado. O novo arquivo será salvo como `anonymised_customer_feedback.csv`.

In [ ]:
arquivo_saida = "anonymised_customer_feedback.csv"
df_anonimo.to_csv(arquivo_saida, index=False)
print(f"Arquivo salvo: {arquivo_saida}")

try:
    from google.colab import files
    files.download(arquivo_saida)
except ImportError:
    print("Fora do Colab: o arquivo foi salvo na pasta atual.")

Arquivo salvo: anonymised_customer_feedback.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Analisar as avaliações com Python

Os cálculos objetivos devem ser feitos primeiro com Python. Depois, uma IA pode ajudar a interpretar ou comunicar os resultados.

In [ ]:
maximo = df_anonimo["Rating"].max()
minimo = df_anonimo["Rating"].min()
intervalo = maximo - minimo
media = df_anonimo["Rating"].mean()

resumo_notas = pd.DataFrame({
    "Métrica": ["Máximo", "Mínimo", "Intervalo", "Média"],
    "Valor": [maximo, minimo, intervalo, round(media, 2)]
})
display(resumo_notas)

print("\nDistribuição por categoria:")
display(df_anonimo["RatingCategory"].value_counts().rename_axis("Categoria").to_frame("Quantidade"))

,Métrica,Valor
0,Máximo,5.0
1,Mínimo,2.0
2,Intervalo,3.0
3,Média,4.0



Distribuição por categoria:


,Quantidade
Categoria,
Alta,10
Média,4
Baixa,1


## 11. Identificar palavras frequentes

A análise abaixo:

1. transforma o texto em minúsculas;
2. extrai palavras;
3. remove palavras muito comuns;
4. conta as palavras restantes.

Como os comentários estão em inglês, a lista de palavras ignoradas também está em inglês.

In [ ]:
stopwords = {
    "the", "a", "an", "and", "or", "to", "of", "in", "on", "with",
    "for", "is", "was", "were", "it", "this", "that", "very", "will",
}

texto_total = " ".join(df_anonimo["Feedback"].dropna().astype(str)).lower()
palavras = re.findall(r"\b[a-zA-Z]{3,}\b", texto_total)
palavras_filtradas = [p for p in palavras if p not in stopwords]
frequencias = Counter(palavras_filtradas).most_common(12)

display(pd.DataFrame(frequencias, columns=["Palavra", "Frequência"]))

,Palavra,Frequência
0,experience,2
1,satisfied,2
2,service,2
3,time,2
4,excellent,1
5,support,1
6,helpful,1
7,staff,1
8,average,1
9,amazing,1


## 12. Categorizar temas dos comentários

Esta classificação é simples e baseada em palavras-chave. Ela serve para praticar a análise, não para substituir uma avaliação mais completa.

In [ ]:
def classificar_tema(texto):
    t = str(texto).lower()
    if any(p in t for p in ["delivery", "delay", "response", "resolution"]):
        return "Prazo e resposta"
    if any(p in t for p in ["support", "staff", "service", "care", "communication"]):
        return "Atendimento"
    if any(p in t for p in ["product", "quality"]):
        return "Produto"
    if any(p in t for p in ["value", "money", "transaction"]):
        return "Valor e processo"
    return "Geral"

df_anonimo["FeedbackCategory"] = df_anonimo["Feedback"].apply(classificar_tema)

print("Notas por categoria:")
display(
    df_anonimo.groupby("FeedbackCategory")["Rating"]
    .agg(["count", "mean", "min", "max"])
    .sort_values("mean")
    .round(2)
)

print("\nCategorias presentes nas avaliações baixas e médias:")
display(
    df_anonimo[df_anonimo["Rating"] <= 3]["FeedbackCategory"]
    .value_counts()
    .rename_axis("Categoria")
    .to_frame("Quantidade")
)

Notas por categoria:


,count,mean,min,max
FeedbackCategory,,,,
Prazo e resposta,3,3.33,3,4
Geral,4,3.75,2,5
Atendimento,5,4.20,3,5
Valor e processo,2,4.50,4,5
Produto,1,5.00,5,5



Categorias presentes nas avaliações baixas e médias:


,Quantidade
Categoria,
Geral,2
Prazo e resposta,2
Atendimento,1


## 13. Engenharia de prompt com dados seguros

Um prompt de análise deve apresentar:

- **papel:** quem a IA deve representar;
- **tarefa:** o que deve ser produzido;
- **contexto:** quais dados seguros estão disponíveis;
- **restrições:** o que não deve ser inventado;
- **formato:** como organizar a resposta;
- **validação:** como indicar incertezas.

### Prompt fraco

> Analise esses dados e diga o que fazer.

### Prompt melhorado

> Atue como analista de experiência do cliente. Analise somente o resumo agregado fornecido. Identifique três tendências, relacione-as às avaliações e sugira três melhorias práticas. Não invente causas que não estejam sustentadas pelos dados. Organize a resposta em uma tabela com: evidência, interpretação, recomendação e limitação.

## 14. Gerar um contexto seguro para o prompt

A próxima célula cria um resumo agregado. Revise o texto antes de copiá-lo para uma ferramenta de IA.

In [ ]:
distribuicao = df_anonimo["RatingCategory"].value_counts().to_dict()
medias_tema = (
    df_anonimo.groupby("FeedbackCategory")["Rating"]
    .mean().round(2).sort_values().to_dict()
)
top_palavras = [palavra for palavra, _ in frequencias[:8]]

contexto_seguro = f'''
Quantidade de registros: {len(df_anonimo)}
Avaliação mínima: {minimo}
Avaliação máxima: {maximo}
Avaliação média: {media:.2f}
Distribuição das categorias: {distribuicao}
Média de avaliação por tema: {medias_tema}
Palavras mais frequentes: {top_palavras}
'''

prompt_final = f'''
Atue como analista de experiência do cliente.

Tarefa:
1. Identifique três tendências sustentadas pelos dados.
2. Explique a importância de cada tendência.
3. Sugira três melhorias práticas para o negócio.

Restrições:
- Use somente o resumo agregado abaixo.
- Não invente causas, percentuais ou informações ausentes.
- Diferencie evidência de hipótese.
- Informe limitações decorrentes do pequeno volume de dados.

Formato:
Apresente uma tabela com as colunas:
Evidência | Interpretação | Recomendação | Limitação.

Resumo agregado:
{contexto_seguro}
'''

print(prompt_final)


Atue como analista de experiência do cliente.

Tarefa:
1. Identifique três tendências sustentadas pelos dados.
2. Explique a importância de cada tendência.
3. Sugira três melhorias práticas para o negócio.

Restrições:
- Use somente o resumo agregado abaixo.
- Não invente causas, percentuais ou informações ausentes.
- Diferencie evidência de hipótese.
- Informe limitações decorrentes do pequeno volume de dados.

Formato:
Apresente uma tabela com as colunas:
Evidência | Interpretação | Recomendação | Limitação.

Resumo agregado:

Quantidade de registros: 15
Avaliação mínima: 2
Avaliação máxima: 5
Avaliação média: 4.00
Distribuição das categorias: {'Alta': 10, 'Média': 4, 'Baixa': 1}
Média de avaliação por tema: {'Prazo e resposta': 3.33, 'Geral': 3.75, 'Atendimento': 4.2, 'Valor e processo': 4.5, 'Produto': 5.0}
Palavras mais frequentes: ['experience', 'satisfied', 'service', 'time', 'excellent', 'support', 'helpful', 'staff']




## 15. Revisar criticamente a resposta da IA

Depois de testar o prompt, responda:

1. A resposta utilizou apenas as informações fornecidas?
2. Algum número foi inventado ou calculado incorretamente?
3. As recomendações estão ligadas às evidências?
4. A IA reconheceu que o conjunto de dados é pequeno?
5. Há generalizações, vieses ou afirmações excessivamente confiantes?
6. O prompt pode ser melhorado? Como?